In [ ]:
import kagglehub
import os
import pandas as pd

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")


print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

csv_path = os.path.join(path, "Q3_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Inspect the first few rows using head()

df.head()

In [ ]:
# Task 3: Display dataset information using info()
df.info()

In [ ]:
# Task 4: Show statistical description using describe()
df.describe()

In [ ]:
df.head()

In [ ]:
# Task 1: Handle missing values appropriately

def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])


  if missing_values.any():
    print("\nHandle Missing Values as needed.")
    df.fillna(df.mean(numeric_only=True), inplace=True)
    #df.fillna(df.mode, inplace=True)

  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# Task 2: Check and remove duplicates if any exist

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")

  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")

  else:
    print("No Duplicate Samples Found.")
check_duplicates(df)

In [ ]:
# Task 3: Encode categorical variables if needed

categories = df.select_dtypes(include=["object"]).columns
#print("Categorical Columns:", list(categories))
categories


In [ ]:
df.head()

In [ ]:
#features = df.select_dtypes(include=["float64"]).columns
#print("Categorical Columns:", list(categories))
#features


In [ ]:
# Task 4: Apply feature scaling to numerical features (Use StandardScaler)

from sklearn.preprocessing import StandardScaler

features = df.columns.drop("Target")
scaler = StandardScaler()

df[features] = scaler.fit_transform(df[features])
df.head()

In [ ]:
# Task 5: Check for target imbalance and state if it is imbalanced or not
import matplotlib.pyplot as plt

def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')
  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)
  plt.show()

check_target_distribution(df, "Target")


In [ ]:
# Task 1: Split the dataset into features (X) and target (y)

X = df.drop("Target", axis=1).astype(float)
y = df['Target'].astype(float)

In [ ]:
y

In [ ]:
# Task 2: Use the correct split: KFold OR StratifiedKFold
from sklearn.model_selection import KFold

n_splits = 5 # K=5 Folds
# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)


In [ ]:
%pip install kagglehub catboost lightgbm tqdm -q

In [ ]:
from catboost import CatBoostRegressor
models = {"CatBoost": CatBoostRegressor(verbose=0)}

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

In [ ]:
def softmax(z):
  z_shifted = z - np.max(z, axis=1, keepdims=True)
  exp_z = np.exp(z_shifted)
  return exp_z / np.sum(exp_z, axis=1, keepdims=True)

In [ ]:
def categorical_cross_entropy(y, y_hat):
  epsilon = 1e-15
  y_hat = np.clip(y_hat, epsilon, 1 - epsilon)
  loss = -np.mean(np.sum(y * np.log(y_hat), axis=1))
  return loss

In [ ]:
def one_hot_encode(y, num_classes):
  y = np.array(y)
  m = len(y)
  # 1. Create a grid of all zeros (num_samples, num_classes)
  one_hot = np.zeros((m, num_classes))
  # 2. Go through each sample one by one
  for i in range(m):
  # Identify which class this sample belongs to
    class_label = int(y[i])
    # In this row (i), set the specific class column to 1
    one_hot[i, class_label] = 1
  return one_hot

In [ ]:
def gradient_descent(X, y, num_classes, lr, n_iters=1000):
# Get the number of samples (m) and number of features (n)
  m, n = X.shape
  theta = np.zeros((n, num_classes))
# One-hot encode the labels
  y_onehot = one_hot_encode(y, num_classes)
  losses = []
  for _ in tqdm(range(n_iters), desc="Training CatBoostClassifier model"):
  # Calculate the logits z
    z = np.dot(X, theta)
    # Get class probabilities using softmax
    y_pred = softmax(z)
    # Compute the gradient of Categorical Cross-Entropy with Softmax
    # ∂J/∂θ = (1/m) * X^T * (y_pred - y_onehot)
    gradient = np.dot(X.T, (y_pred - y_onehot)) / m
    # Update weights
    theta -= lr * gradient
  # Track loss
    loss = categorical_cross_entropy(y_onehot, y_pred)
    losses.append(loss)
  return theta, losses

In [ ]:
for fold_idx, (train_index, test_index) in enumerate(kf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")
  # Get the train & test split for this fold
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]
  # Train using gradient descent with learning rate = 0.5
  theta, losses = gradient_descent(X_train, y_train, lr=0.5, num_classes=2)
  # Calculate z & class probabilities for X_test
  z = np.dot(X_test, theta)
  y_pred_proba = softmax(z)
  # Pick the predicted classes with the highest probability
  y_pred = np.argmax(y_pred_proba, axis=1)
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, average='macro') # for multiclass f1 score, y
  # Store results
  re['loss'].append(losses)
  re['acc'].append(accuracy)
  re['f1'].append(f1)

In [ ]:
re = {}
for model_name in models:
  re[model_name] = {'accuracy': [], 'f1':[]}

In [ ]:
for fold_idx, (train_index, test_index) in enumerate(kf.split(X, y)):

  print(f"\nFold {fold_idx + 1}/{n_splits}")
  # Get the train & test split for this fold

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  theta, losses = gradient_descent(X_train, y_train, lr=0.5, num_classes=4)
  # Calculate z & class probabilities for X_test
  z = np.dot(X_test, theta)
  y_pred_proba = softmax(z)
  # Pick the predicted classes with the highest probability
  y_pred = np.argmax(y_pred_proba, axis=1)

  # Train & Validate Models
  for model_name, model in models.items():
    print(f"Training {model_name}...")
  # Fit the model on train data
    model.fit(X_train, y_train)
  # Use the model to predict the test data
    y_pred = model.predict(X_test)
  # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    #f1 = f1_score(y_test, y_pred, average='micro') # for multiclass f1 score,
    results[model_name]['accuracy'].append(accuracy)
    #results[model_name]['f1'].append(f1)

In [ ]:
# Task 2,3,4,5: Write your code here:

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

for model_name, model in models.items():
  print(f"Training {model_name}...")
  # Train
  model.fit(X_train, y_train)
  # Predict
  y_pred = model.predict(X_test)


  # Calculate  metrics
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, average='macro')

  results['acc'].append(accuracy)
  results['f1'].append(f1)

In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: